## 🧭 用 SDD + TDD 驯服 AI：让「写得快」也「信得过」

> 模块 7「AI Coding 带来的范式变革」· 第 1 课

腾讯 2025 年内部研发大数据报告里有两组数字：超过九成的工程师在用 CodeBuddy 做 AI 辅助编程，AI 生成的代码占新增代码的 50%，而 2024 年这个比例是 35%。百度那边，李彦宏在 Create 2025 大会上给出的数字是文心快码生成了超过 43% 的新增代码。在大厂内部，AI 写代码已经不是个别人在试验，而是整个工程团队的日常工作状态。

YC 的 Garry Tan 发过一条推文，说 2025 年冬季批次里，四分之一的创业公司代码库 95% 的代码行是由 AI 生成的。原话是："That's not a typo. The age of vibe coding is here."

这几个数字拼在一起，指向同一件事：AI 大规模参与编码已经不是预测，是正在发生的日常。

不过 YC 管理合伙人 Jared Friedman 在后面补了一句很重要的限定。他说 95% 的代码由 AI 写，不等于 95% 的工程工作消失了。打字和初版生成的负担确实被拿走了，审查、架构、调试、测试、安全、维护的负担都还在。

Karpathy 本人的轨迹也值得注意。"vibe coding"这个词是他 2025 年 2 月提出来的，当时的意思是在周末玩具项目里"完全交给感觉、忘掉代码这回事"。后来他自己把这个词收了回去，改叫"agentic engineering"——有严格监督和验证的 AI 工程。

本文讨论的正是这条线：在 AI 已经能大量产出代码的前提下，怎么确认它写得对、敢不敢上线。核心工具是两个——SDD（规格驱动开发）和 TDD（测试驱动开发）。

> ℹ️ **讨论**：最近一次 AI 写的代码上线后出 bug，回过头看——当时是少了一条明确的规则（没有 spec），还是少了一条能拦住它的测试（没有 TDD）？聊天区扣 1 或 2，也可以文字描述当时的情况。

### 🎯 核心内容

**一、SDD + TDD：把"对不对"变成可以交给机器的事**

- SDD（规格驱动开发）回答"什么算对"——一份人能在几分钟内审完的规格
- TDD（测试驱动开发）把"对"变成机器能判的东西——先写测试（全红），再补实现跑到全绿
- 两者配合，把概率系统约束到可接受区间

**二、用什么工具：coding agent 选型**

- 选型第一轴是 spec 遵从度，其次才是自主探索、成本、中文
- 课上的演示用 Codex 或 Claude Code

**三、从头走一遍：spec → 后端 → 契约 → 前端**

- 从一份待办事项 API 的规格出发，spec 探讨 + TDD 红绿 → 后端自动产出 OpenAPI 契约
- 契约变成前端的规格——确定性沿技术栈一层一层往下传导

**四、从一次性流程到工程体系**

- 把手工流程固化成可重复流水线（L3、多智能体协作、CI/CD 门禁）
- 让 AI 不每次都从零开始（Rules + Spec + Skills 三种沉淀，破解"代码考古"）
- 写完谁负责（AI 自我审查 + 人最终决策，从"超级个体"到"超级团队"）

### 💡 1. SDD + TDD：把"对不对"变成一个可以交给机器的事

#### 1.1 SDD（规格驱动开发）是什么

SDD，即规格驱动开发（Spec-Driven Development），核心思路一句话：**规格是首要产物，代码是从规格构建出来的。**

传统开发流程里，需求文档写完，开发就开始写代码。规格——如果写的话——往往夹在需求文档的段落里，或者散落在各个接口文档中，不是独立的一级产物。SDD 把这个顺序倒了过来：在动手写任何实现代码之前，先把"做成什么样算对"写成一份独立、完整、人能审的规格，审过之后，代码从规格生成。

规格阶段有一个重要的设计原则：**聚焦"怎么用"，不急于讨论"怎么实现"。** 好的 spec 讨论的是场景——在什么情况下、解决什么问题、输入和输出是什么。它不讨论用什么框架、数据库怎么设计、服务怎么拆分。

以设计一个待办事项 API 为例，spec 阶段应该问的是：用户创建待办时要不要填截止日期？标记完成之后还能不能撤销？列表默认按什么排序？——这些是"怎么用"的问题。后端用 FastAPI 还是 Flask、数据存内存还是 SQLite，这些是实现方案的范畴，在 spec 确认之后再回答。

同样的道理，在设计一个框架的时候，第一件事不是画架构图，是先写出预期的调用代码样例——这段代码读起来是不是顺畅、使用者需要写多少行才能完成一个常见任务。代码样例确认了"易用"，内部的实现方案是后续顺水推舟的事情。这就是 spec 阶段的核心价值：用最轻的方式把"怎么用"聊清楚、定下来，之后的实现方案和测试用例都是自然推导。

这份规格通常包含：数据模型、接口的输入和输出、校验规则、边界条件、错误返回，以及明确不做的事。它的核心特征是**人可以快速审完**——不是上百页的文档，而是十几条可以逐条对照的约束。

SDD 这个方向在 2025 年前后加速成型，直接原因是 AI 编码工具的普及。当代码的"写"越来越快、越来越便宜，"写得对不对"就成了瓶颈。GitHub 在 2025 年 9 月开源了 Spec Kit（specify → plan → tasks → implement），AWS 在 2025 年 7 月发布了 Kiro（需求 → 设计 → 任务 → 实现）。它们面向的是同一个问题：AI 生成的代码怎么从凭感觉走向可验证。

#### 1.2 TDD（测试驱动开发）是什么

TDD，即测试驱动开发（Test-Driven Development），是 Kent Beck 在 1990 年代末系统化提出的一套开发方法。核心操作是一个短循环：**先写一个会失败的测试（红），再写刚好能让它通过的最少实现（绿），然后清理代码（重构）。**循环一次只走一个小步，每一步都有测试兜底。

TDD 在提出后的二十多年里，认同的人多、坚持的人少。原因很简单：手写测试太慢了。在快速迭代的项目里，测试经常是第一个被砍的。

AI 把这个账重新算了一遍。AI 写测试的速度比人快得多——原来一个下午才能写完的测试，现在十分钟就能生成。而且测试恰好是 AI 生成代码最缺的东西：一个不带感觉、只判断对错的裁判。所以在 AI 编码的时代，TDD 从"道理对但太慢"变成了"道理对且现在做得起了"。

这里值得补一个判断。TDD 这套方法本质上是**反人性**的——它要求人在写实现之前先写一堆会失败的测试，把满足感最强、收益最直接的"先把功能跑起来"往后推。人会嫌麻烦、会偷懒，本能地先做那些能立刻看到成果的事，测试于是一拖再拖。这不是纪律问题，是人性。

但同样这套约束，放到 AI Agent 身上就**刚刚好**。AI Agent 不会嫌麻烦，也不在乎满足感来得早还是晚，它真正需要的恰恰是 TDD 能给的两样东西：明确的行为指导（规格和测试告诉它要做成什么样），和清晰的检验标准（测试的红绿告诉它做到了没有）。对人来说是负担的东西，对 AI 来说是它最需要的脚手架。从这个角度看，TDD 在 AI 时代的复活，不只是因为写测试变快了，更是因为执行这套方法的主体——从一个会偷懒的人，换成了一个需要明确指令才能干好活的 Agent。

#### 1.3 SDD 与 TDD 怎么配合

两个概念经常被放在一起提，但它们管的是不同的事：

- **SDD 回答"什么是对"**：规格是人确认的，描述的是"做成什么样算对"
- **TDD 把"对"变成机器能判的东西**：规格的每一条规则和边界，变成一条可执行的测试

规格是源头，测试是规格的可执行形态。两者配合的流程是：

1. 先写规格（SDD），人审过
2. 按规格写测试（TDD 的红），此时没有实现，全红
3. 补实现跑到全绿（TDD 的绿），测试没绿就继续修

这套流程走到"全绿"，代码就算是有了第一层保障。具体到 Python 技术栈，验收可以分成三层：

- **第一层：pyright 静态校验。** 类型错误、缺少导入、参数不匹配——这些不需要跑测试，静态分析就能拦下来。每次 AI 产出代码之后，先跑 pyright，零 error 再往下走
- **第二层：pytest 全量用例。** 这是 TDD 的主体——spec 的每一条规则和边界都变成一条测试，全绿才是"行为正确"
- **第三层：example 实际案例 + 人工复检。** 前两层是机器判的，但有些东西机器判不了——实现方案是不是好用（调用方式是否简洁、错误提示是否清晰）、业务逻辑是不是真的正确（case 覆盖的语义对不对）。这一层要人来看，关注的是易用性和业务正确性

三层下来，效果是把一个概率系统（LLM）约束到可接受区间：

![SDD 定义“什么是对”，TDD 把“对”变成机器能判，一起把概率产出夹回可接受区间](assets/01_sdd_tdd.drawio.svg)

#### 1.4 没有这套东西的时候，发生了什么

小李在一家电商公司做后端。上周他让 AI 写了一段处理订单金额的函数——逻辑看起来通顺，变量命名也对，读着差不多。合进去，上线。

两周后线上出了一笔异常单：一个订单同时命中会员折扣和优惠券，两个优惠叠加了，金额算错。小李回来翻代码，发现在 `calculate_discount` 分支里，AI 根本没有处理"会员身份同时持有有效优惠券"的冲突逻辑——边界条件被静默吞掉了。

问题出在哪一步？不是 AI 的能力不够。是小李 review 这段代码的时候，手里没有一个可以逐条对照的标准。他凭的是感觉——"读着像对的"。LLM 的产出是概率分布，同一个 prompt 两次输出不完全一样。如果 review 的标准也是模糊的，那审查就成了碰运气：产出碰巧像对的就放过去，碰巧不像才拦下来。这种没有可对照标准、靠感觉判断对错的写法，常被叫作"感觉式编程"——SDD 要解决的正是它。

SDD 做的事就是给小李一个可以逐条对照的东西。十几条规格——标题不能为空、长度不超过 100、去空白后再判空——审完不需要五分钟。如果当初有这份规格，那条"会员和优惠券不能叠加"的规则在审规格的时候就暴露了，不会等到上线两周后。

TDD 做的事是让这些规则不会被"忘了测"。每条规格对应一条测试——空标题、仅空白、超长、不存在 id、重复标记——八条测试全红的时候，AI 得一条一条修到绿。少一条绿不过，少一条就没测到。

#### 1.5 SDD 的核心操作与行业形态

SDD 的操作可以总结成三步——有人把它叫作"Spec Coding 三铁律"。"铁律"这个说法偏重，但三步的顺序值得记住：**先写规格、规格必须可验证、小步可回滚。**

先写规格——在让 AI 写实现之前，先把输入、输出、边界、校验规则、明确不做的范围写成一份人能审完的文档。审规格审的是"对不对"，不是"看着像不像对的"。

规格必须可验证——每一条规则都要能变成一条测试或一个类型约束。不可验证的规格等于没有——"性能不能太差"是不可验证的，"P99 延迟不超过 200ms"才是。

小步可回滚——每次只改一件能独立验证的事。一个端点一个红绿循环，不是一次性把整个服务写完。每步的爆炸半径控制在一个端点以内。

行业里已经有一些具体形态在往这个方向走。GitHub Spec Kit 把流程固化成 specify → plan → tasks → implement，AWS Kiro 内置了需求 → 设计 → 任务 → 实现四个阶段。它们不是某一个工具的专利，是 AI 代码从"凭感觉"走向"可验证"的共识方向。

### 📊 2. 用什么工具：coding agent 选型

理解了 SDD+TDD 这条线之后，有一个很实际的问题：用什么工具来做？下面的演示会在一个 coding agent 上完整跑一遍——课上用的是 Codex 或 Claude Code。在看它跑之前，先把选型的判断框架讲清楚。

同一套 SDD+TDD 流程，在不同 coding agent 上的表现是有差异的。如果押注的是这条线——先规格、先测试、再实现——选型时第一个要看的就不是"谁更聪明"或"谁代码量更大"，而是**对 spec 的遵从度**：给它一份规格，它是老老实实照着做，还是中途自由发挥。其他维度（自主探索、审美、成本、速度）放在这一条之后。

下面是实际使用中的一手体感。工具迭代很快，这里的判断随时可能过时（截至 2026-05）。

| 维度 | Codex | Claude Code 原厂 | CC + DeepSeek | GitHub Copilot |
|---|---|---|---|---|
| 复杂任务 / spec 强遵从 | **最好** | 强 | 可能有偏差 | 不推荐 |
| 自主探索能力 | 中 | **最强** | 中 | 弱 |
| UI 审美 / 文科类任务 | 中 | **好** | 中 | — |
| 中文文案语言顺畅度 | 中 | 中 | **好很多** | — |
| 成本 | 较高 | 较高 | **便宜** | — |
| 速度 | 快 | 中 | **耗时长** | — |
| 整体交付质量 | 高 | 高 | 可以（够用） | 低端平替 |

**Codex**：复杂任务、强 spec 遵从场景下的首选体验。在需要严格跟着 spec 走的场景里（spec→test→实现），偏离度低。

**Claude Code 原厂**：自主探索能力最强，UI 审美和文科类任务表现好。适合需要 AI 主动探索、自由发挥的场景。

**CC + DeepSeek**：性价比路线。复杂任务和 spec 遵从比原厂略有偏差，耗时长一些，但整体交付质量够用。便宜，而且中文文案的顺畅度明显更好。

**GitHub Copilot**：以前是低端平替，现在不推荐。Pro 对大陆新用户有限制，试用政策之前反复过。政策类信息易过期，使用前自己再确认。

这张表不是排名。按自己的场景填权重即可——强 spec 复杂任务就押 Codex，中文多预算紧就押 CC+DS，要自主探索和审美的就押 CC 原厂。接下来的演示，就用其中一个工具，跑一遍从规格到代码的完整流程。

### 🧩 3. 命题：待办事项 API · 规格

要实际演示 SDD+TDD 的完整流程，需要一段足够具体、又不至于太大的需求。下面是一份待办事项 API 的规格——从需求出发，先把"做成什么样算对"定下来。

读完这份规格大概需要两分钟。读完之后可以问一个问题：如果 AI 按它写出来、测试也照它通过——这个服务应该就是对的吗？如果答案是"是"，那这份规格就是有效的。

**数据模型**

| 字段 | 类型 | 约束 |
|---|---|---|
| `id` | int | 服务端自增，从 1 开始，只读 |
| `title` | str | 必填，去掉首尾空白后非空，长度 1–100 |
| `done` | bool | 默认 `false` |

**端点**

| 端点 | 成功 | 失败 |
|---|---|---|
| `POST /todos` 新建待办 | `201`，返回完整 Todo（含 `id`、`done=false`）| title 空 / 仅空白 / 超 100 字 → `422` |
| `GET /todos` 列出 | `200`，按 id 升序 | — |
| `POST /todos/{id}/done` 标记完成 | `200`，`done=true`（幂等，重复标记仍返回 200）| id 不存在 → `404` |

**边界（每一条之后会变成一条测试）**

- 空标题 `""` → 422
- 仅空白标题 `"   "` → 422（去首尾空白后为空，不能靠它绕过校验）
- 超长标题（101 字）→ 422
- 标记不存在的 id（如 999）→ 404
- 重复标记完成：幂等，仍然返回 `done=true`、200

**不在这次的范围（划死，防止 AI 自行发挥）**

不做删除、不做编辑标题、不做分页、不做持久化（内存存储即可）、不做鉴权。

划死边界是 SDD 的一个关键动作：一份规格必须说清楚做到哪里算结束、什么是明确不做的。否则 AI 可能会"顺便加一个删除功能"。

> 完整版：[code/backend/spec/todo_api_spec.md](code/backend/spec/todo_api_spec.md)。

### 🔧 4. 环境准备

In [ ]:
# 后端运行环境
# pip install -r code/backend/requirements.txt
# 主要依赖：fastapi、uvicorn、httpx、pytest

> ℹ️ **接下来的演示**：下面用一份真实的 spec，从零走一遍 SDD+TDD 的完整流程。留意三个节点——① spec 探讨时改了哪些地方、② 红绿循环中怎么回喂失败信息、③ 最后产出的 OpenAPI 契约是怎么变成前端约束的。

> 注：标了 🔴 的代码片段不提前写好——由 spec 驱动，用 AI 现场生成。每一步产出的代码都来自规格，不是手工编写的参考答案。

### 🔴 5. 从 spec 到可运行的后端：spec 探讨 → TDD 红 → TDD 绿

#### 步骤 1：spec 探讨

打开上面的规格，先过一遍——改几处需要补充的地方。比如把标题长度加上"上限 100 字"，把"标记完成"补上"重复标记是幂等的"。这些补充来自对业务的理解：数据库字段长度总有一个合理值，用户连续点两次"完成"也不应该报错。

规格是人在审、人在改。AI 没有替代这个判断，只是把判断之后的实现加速了。

#### 步骤 2：TDD 红——按规格只写测试

把确认后的 spec 喂给 AI，要求它只写测试、不写实现。需要明确几点：

- 按 spec 写测试，不要写实现代码
- 三个端点各至少一条正常路径的测试
- spec"边界"小节的每一条对应一条测试
- 每条测试尽量独立，不依赖前一条的数据库状态

跑一下——全是红的，因为没有实现。按照 spec 的边界数量，8 条测试全部 FAIL。

测试现在是退出标准：AI 得把这些红全部变成绿，才算完工。

In [ ]:
# 🔴 现场 AI 生成的测试文件在这里运行
# !cd code/backend && python -m pytest -q
# 只有测试、没有实现时，预期全部 FAILED

#### 步骤 3：TDD 绿——补实现，红转绿

让 AI 补实现。一个端点一个端点来，不要一次堆给它——这刚好是对应"渐进式复杂度管理"的实操。

中间有一个操作值得单独说明。可以故意留一个边界让它先没处理，比如空白标题 `"   "`（去掉首尾空白后其实是空的，但 AI 容易只判断了 `""` 而漏掉 `"   "`），对应的测试是红的。

这时候怎么告诉它？把失败的断言、输入和期望原样喂回去。例如：

> `test_blank_title_rejected_422` 这个测试没通过。输入是 `{"title": "   "}`，期望返回 422 校验错误，但实际返回了 201。校验逻辑需要先 strip 再判空。

AI 拿到具体失败信息，一两轮就能定位修复。笼统说"有 bug"和给具体断言，修复效率差好几倍——前者 AI 得猜问题在哪，后者直接定位。最后全绿——8 passed。

In [ ]:
# 🔴 AI 补完实现后跑绿
# !cd code/backend && python -m pytest -q
# 预期：8 passed

#### 关键观察

- **审查成本被前移、被压缩**。全程审的是 spec（人改的）和测试（机器生成的，能快速扫结构），不是上百行实现。AI 的实现是在测试的红绿之间自己迭代出来的——不需要逐行去读它改了哪里，测试替代了这个工作
- **测试是退出标准**。AI 不是"觉得写好了"，是机器判它过了。从红转绿的过程中，AI 从概率引擎变成了有明确 stop condition 的执行者
- **渐进式复杂度**。一个端点一个红绿循环——新建完成再写列出，列出完成再写完成标记。不是一次把三个端点全甩给它。每一步的爆炸半径在一个端点以内
- **失败信息要结构化回喂**。不是"fix the bug"，是把具体的断言、输入、期望和实际返回一起给。定位错误的时间远少于让它自己猜

上面走通了一条线：从一份人能审完的规格出发，经过测试驱动，长出了一个可运行的后端服务。接下来是第二步——后端跑起来之后，它自动产出的一份契约，能不能继续约束下一层。

> ℹ️ **讨论**：刚才这段流程里，人花时间最多的环节是什么？是审 spec 那两分钟、是回喂失败信息那次、还是别的？聊天区说一下——这能帮每个人判断自己目前 SDD+TDD 里最卡的地方在哪。

### 🔴 6. 后端契约 → 前端的规格

后端跑起来后，FastAPI 自动生成了一份 OpenAPI 契约（`http://127.0.0.1:8000/openapi.json`）。打开看：

- `paths` 里三个端点，没有任何多余的东西
- `TodoCreate` schema 只有 `title` 一个字段
- `title` 上挂着 `minLength: 1`、`maxLength: 100`

这是机器能读的数据，不是给人看的文档。它是一个可以被传递的硬约束。

#### 步骤 4：把契约喂给 AI 生成前端 UI

把这份 `openapi.json` 提供给 AI，让它生成待办列表和新建表单的页面。生成完后，逐条看里面的约束从哪来的：

| 前端里看到的 | 来自契约哪一项 |
|---|---|
| 页面只调 `/todos`、`/todos/{id}/done` | 契约的 `paths`——总共就这些端点 |
| 新建请求体只发 `{ title }` | `TodoCreate` schema——只有 `title` 一个字段 |
| 标题输入框 `maxlength="100"` | `title.maxLength` |
| 标题为空不允许提交 | `title.minLength: 1` |

前端没有自己拍脑袋决定任何东西。所有约束全部从后端契约顺下来。AI 想编一个后端不存在的字段都编不出来——输入就是契约，没有别的东西可以参照。

#### 关键观察

**API 定义不是文档——是上一层的产出变成下一层的规格。** 确定性不是一锤子买卖，可以一层一层往下传导。

![spec 沿技术栈向下游传导：spec 探讨→TDD红绿→后端→OpenAPI 契约→前端 UI](assets/02_spec_propagation.drawio.svg)

- 后端产出的 OpenAPI，是前端的 spec
- 前端从这个 spec 长出来，端点、字段、类型、长度全是后端定的
- 后端加一个字段 → 契约自动变 → 前端重新生成时自动跟上

这不是"沟通对齐"，是代码级的硬约束。甚至可以更进一步——从 OpenAPI 直接生成 typed client 或 mock，把约束落到类型系统里。前后端联调的对错，在契约层面就被拦住了，而不是联调的时候才发现。

SDD 不是某个工具的专利，是这个方向的共识。不管载体是 OpenAPI、GraphQL schema 还是 gRPC proto，本质一样：机器可读的契约，让确定性可以沿技术栈往下走。

### ⚙️ 7. 把手工流程固化成可重复的流水线

上面是人手动一步步带的。一个自然的延伸是：这套流程能不能固化下来，变成每次都能自动跑的通路？

有一个被提起很多的说法叫 L3 AI Coding，把 AI 编程的自动化程度分了几个层级——从 IDE 补全、对话式编程、到 Agent 自主规划执行、再到更完整的自动化。它不是一个正式标准，但可以帮人定位当前在哪个层级、下一级需要补什么。

另一个方向是多智能体协作。把"规划 / 实现 / 审查"三个角色拆给不同的 Agent，各自维护独立的上下文，互相交接的不是一段超长对话，而是规格、diff、测试结果这些产物。

还有一个容易被忽略的点：AI 产出的代码应该和人走同一道 CI/CD 门禁。类型检查、测试、lint——不管代码是谁写的，过不了就是过不了。AI 能写得更快，但质量门槛不应该比人低，否则速度优势会逐渐变成质量债。

![手工流程固化成流水线：规划/实现/审查 Agent 分工 + CI 门禁，AI 无豁免权](assets/03_pipeline.drawio.svg)

上面演示的是轻量版。把这条链路完整搬到 GitHub 上——issue 指派给 coding agent、自动开 PR、CI 跑门禁、自动 AI 评审、最后由人按 merge——是后面章节的内容。

> ℹ️ **过渡**：流水线让流程可以重复跑。但有一个前提——每次跑的时候，AI 要知道这个项目的规则是什么。下一节讨论怎么把规则沉淀下来。

### 📦 8. 让 AI 不用每次都从零开始

每开一个新会话，AI 又把项目当陌生人，从头猜。上次踩过的坑——函数命名用 snake_case、测试放在 tests/ 下、不新增没写在 requirements 里的依赖——下次还会再犯。

前面确认的那份 spec，本身就是一个可以一直留着的东西。固定住它，下次 AI 就不用重学——拿出这个模块的规格，AI 就有了"上次做成什么样算对"的全部记忆。

顺着这个思路，有三类东西值得持续沉淀——把它们放在一起，就是常说的 **Rules + Spec + Skills 三位一体**：

| 沉淀什么 | 管什么事 | 直接效果 |
|---|---|---|
| **Rules** | 约束类（命名规范、目录约定、禁止项、不可逆操作的审批规则）| 产出贴着团队的工程口味，不需要每个 PR 手动纠风格 |
| **Spec** | 意图类（模块要解决什么、边界在哪、明确不做什么）| AI 理解的是"为什么"，不是照抄上次的代码 |
| **Skills** | 可复用动作（固化的常用操作流程，如"给新数据表生成 CRUD + 测试"的步骤）| 不用每次重新拼装步骤 |

把散在人脑、PR 评论、群聊里的口头知识固化成 AI 能直接读的文本——新人和 AI 都不用再翻聊天记录做"代码考古"。这个过程越用越省力，沉淀越多、复用越快——有人把这种正反馈叫作"知识飞轮"。落到日常就一句话：每确认一份规格、每踩一次坑、每总结一条规则，都在垫高团队和 AI 下一次的起点。

团队级的 spec 管理体系在后面章节会细讲。这里先确立一个习惯：每次 AI 产出跑偏，不只是在 PR 里改掉就完了，而是把这条约束写进 Rules 或 Spec——让它下次不会再犯。

### ✅ 9. 写完以后，谁来担保它是对的

测试全绿不等于可以直接上线。中间还差一步——谁来担保这段代码放进生产环境不会出问题？

**第一道关：AI 自我审查（自审）。** 换一个会话或让 AI 换审查视角，把产出丢给它，让它在几个具体方向上找问题：边界有没有处理、异常路径上有没有静默吞错、和 spec 有没有偏差、哪些行为缺测试。它能抓到很多"明显错"——没处理的 None、漏掉的异常分支、被测漏的边界。

但它抓不住一类问题：**代码完全正确，但理解错了需求**。因为 AI 不知道真实的业务意图。

**第二道关：人最终决策。** 不可逆的操作（删数据、动钱、对外发布、改线上配置）、业务语义对不对——这些没有机器能替人判断，必须在合并之前由人按确认。两关都放在合并前，而不是上线后补救。

整条链路：

![两道审查关：AI 自审抓明显错，人决策抓和意图不符与不可逆操作，都在合并前](assets/04_review.drawio.svg)

这套两道关是**个人和小队尺度**的做法——一段审查 prompt 加一份合并前 checklist 就能跑起来。到了大型项目，"我审一下"不够了，需要把它系统化成不依赖某个人盯着的质量体系：生成 → 验证 → 修正的闭环、用测试做判据自动把关——这是下一节课要展开的内容。

AI 先放大了个体——一个人借助它能顶过去一个小组的产出，这就是所谓的"超级个体"。但如果每个人的方法、规格、踩坑经验全锁在自己脑子里，一堆超级个体并不等于一个"超级团队"——一个人休假，他的 prompt、他的标准也跟着休假，团队反而更依赖个人。怎么从"超级个体"走到"超级团队"，后面两节课会从工程管理和组织分工两个角度展开。

### 📌 10. 总结

回到开头那个判断：AI 已经拿走了"写"的负担——打字、初版生成、重复性改动。但审查、架构、调试、测试、安全、维护的负担都还在。

怎么接住它们：

- **SDD** 定义什么是对——一份人能在几分钟内审完的规格
- **TDD** 把"对"变成机器能判的东西——AI 从红迭代到绿才算完
- **API 契约** 让确定性不在后端停下来——它变成前端的 spec，一层一层往下传导。确定性不是一锤子买卖，是可以逐层传递的

可以延伸思考的一个问题：在自己的项目里，**哪一层的产出可以当下一层的 spec？** 找到了，就有一条从源头往下的约束链。

下一章把这套思路搬到大型项目里：先让 AI 通读老代码产出 spec 和特征测试（锁住现有行为），再在测试兜底下安全重构——改完，测试还是绿的，证明行为没变；然后讨论怎么用 GitHub 的 issue → PR → commit → workflow 把一群人协作的 AI 管线串起来。

> ℹ️ **课后讨论**：在自己的项目里找一层产出——它能不能变成下一层的 spec？比如后端 API 的 OpenAPI 契约能不能约束前端、数据库 schema 能不能约束 DAO 层、protobuf 能不能约束上下游服务。找到了发聊天区——找不到也发，说说你判断它"不能"的原因（是这层不产生可传递的约束，还是约束在、只是没写成机器能读的形式）。

### 📎 附录 A：带走物清单

以下模板和工具卡片可以在实际项目中直接使用：

| 文件 | 用途 |
|---|---|
| [01_spec模板](materials/01_spec模板.md) | 让 AI 写代码前，先填规格 |
| [02_测试先行清单](materials/02_测试先行清单.md) | TDD 红绿循环的操作顺序 |
| [03_API契约约束前端_方法卡](materials/03_API契约约束前端_方法卡.md) | 用 OpenAPI 约束前端的方法 |
| [04_coding-agent选型对照表](materials/04_coding-agent选型对照表.md) | 按方法选工具 |
| [05_Rules模板_AGENTS](materials/05_Rules模板_AGENTS.md) | 项目 Rules 起步模板 |
| [06_审查视角prompt与合并前checklist](materials/06_审查视角prompt与合并前checklist.md) | AI 自审 + 人审两关 |